In [ ]:
# ambiente local (celula 1 do notebook)

import os
import random
from calendar import monthrange
from datetime import date, datetime, timedelta
from decimal import Decimal, InvalidOperation
from numbers import Integral
from zoneinfo import ZoneInfo

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

ambiente = (os.environ.get("AMBIENTE") or "MODELAGEM").strip().upper()
if ambiente == "MODELAGEM":
    load_dotenv("desenv.env", override=False)

DB2_CHUNKSIZE = 10_000
print("Ambiente Python local preparado.")


In [ ]:
# contratos e adaptador local do DB2 (celula 2 do notebook)

class ErroPipeline(Exception):
    def __init__(
        self,
        codigo,
        mensagem,
        etapa=None,
        objeto=None,
        detalhes=None,
        acao=None,
    ):
        super().__init__(mensagem)
        self.codigo = str(codigo).strip() or "ERRO_PIPELINE"
        self.mensagem = str(mensagem).strip()
        self.etapa = str(etapa).strip() if etapa else None
        self.objeto = str(objeto).strip() if objeto else None
        self.detalhes = dict(detalhes or {})
        self.acao = str(acao).strip() if acao else None

    def __str__(self):
        partes = [f"[{self.codigo}] {self.mensagem}"]
        if self.objeto:
            partes.append(f"objeto={self.objeto}")
        return " | ".join(partes)


class ErroContratoDados(ErroPipeline):
    pass


ultima_leitura_db2 = {
    "linhas": 0,
    "memoria_estimada_bytes": 0,
    "chunks": 0,
}


def executar_select(sql, params=None, chunksize=None):
    global ultima_leitura_db2

    metodo = getattr(cliente_db2, "query", None)
    if not callable(metodo):
        raise ErroContratoDados(
            codigo="DB2_QUERY_INDISPONIVEL",
            mensagem="O cliente DB2 nao disponibiliza query().",
            etapa="CONEXAO",
            objeto="cliente_db2",
        )

    ultima_leitura_db2 = {
        "linhas": 0,
        "memoria_estimada_bytes": 0,
        "chunks": 0,
    }
    try:
        resultado = metodo(
            str(sql),
            params=list(params) if params else None,
            chunksize=chunksize,
        )

        if isinstance(resultado, pd.DataFrame):
            partes = [resultado.copy()]
        else:
            if isinstance(resultado, (str, bytes, dict)) or resultado is None:
                raise ErroContratoDados(
                    codigo="DB2_RETORNO_INVALIDO",
                    mensagem="A consulta DB2 deve retornar Pandas ou chunks Pandas.",
                    etapa="CONEXAO",
                    objeto="Db2.query",
                )
            try:
                partes = []
                for parte in iter(resultado):
                    if not isinstance(parte, pd.DataFrame):
                        raise ErroContratoDados(
                            codigo="DB2_CHUNK_INVALIDO",
                            mensagem="Um chunk DB2 nao e DataFrame Pandas.",
                            etapa="CONEXAO",
                            objeto="Db2.query",
                        )
                    partes.append(parte.copy())
                    ultima_leitura_db2["linhas"] += int(len(parte))
                    ultima_leitura_db2["memoria_estimada_bytes"] += int(
                        parte.memory_usage(index=True, deep=True).sum()
                    )
                    ultima_leitura_db2["chunks"] += 1
            except TypeError as exc:
                raise ErroContratoDados(
                    codigo="DB2_RETORNO_INVALIDO",
                    mensagem="A consulta DB2 deve retornar Pandas ou chunks Pandas.",
                    etapa="CONEXAO",
                    objeto="Db2.query",
                ) from exc

        if not partes:
            frame = pd.DataFrame()
        elif len(partes) == 1:
            frame = partes[0]
        else:
            colunas = list(partes[0].columns)
            if any(list(parte.columns) != colunas for parte in partes[1:]):
                raise ErroContratoDados(
                    codigo="DB2_CHUNKS_INCOMPATIVEIS",
                    mensagem="Os chunks DB2 possuem colunas diferentes.",
                    etapa="CONEXAO",
                    objeto="Db2.query",
                )
            frame = pd.concat(partes, ignore_index=True, copy=False)

        frame.columns = [str(coluna).upper() for coluna in frame.columns]
        ultima_leitura_db2 = {
            "linhas": int(len(frame)),
            "memoria_estimada_bytes": int(
                frame.memory_usage(index=True, deep=True).sum()
            ),
            "chunks": max(ultima_leitura_db2["chunks"], len(partes)),
        }
        return frame
    except MemoryError as exc:
        raise ErroContratoDados(
            codigo="VOLUME_LOCAL_INCOMPATIVEL",
            mensagem="O recorte excedeu a memoria disponivel para Pandas.",
            etapa="TRANSACOES",
            objeto="TRAN_RLZD_INST_PCT",
            detalhes=dict(ultima_leitura_db2),
        ) from exc
    except ErroContratoDados:
        raise
    except Exception as exc:
        raise ErroContratoDados(
            codigo="DB2_CONSULTA_FALHOU",
            mensagem="Falha na consulta direta ao DB2.",
            etapa="CONEXAO",
            objeto="Db2.query",
            detalhes={"tipo_erro": type(exc).__name__},
        ) from exc


In [ ]:
# conexao DB2 local (celula 3 do notebook)

from bbmagic.db2 import Db2

nomes_configuracao = (
    "DB2_USER",
    "DB2_PASSWORD",
    "DB2_HOST",
    "DB2_DATABASE",
    "DB2_PORTA",
)
configuracao_db2 = {
    nome: (os.environ.get(nome) or "").strip()
    for nome in nomes_configuracao
}
ausentes_db2 = [
    nome for nome, valor in configuracao_db2.items() if not valor
]
if ausentes_db2:
    raise ErroContratoDados(
        codigo="CONFIG_DB2_AUSENTE",
        mensagem="Configuracao obrigatoria do DB2 nao informada.",
        etapa="CONEXAO",
        objeto=", ".join(ausentes_db2),
        acao=(
            "Preencher desenv.env ou as variaveis do ambiente de Modelagem."
            if ambiente == "MODELAGEM"
            else "Configurar externamente as variaveis obrigatorias na plataforma."
        ),
    )

try:
    porta_db2 = int(configuracao_db2["DB2_PORTA"])
    if porta_db2 <= 0:
        raise ValueError
except ValueError as exc:
    raise ErroContratoDados(
        codigo="CONFIG_DB2_PORTA_INVALIDA",
        mensagem="DB2_PORTA deve ser um inteiro positivo.",
        etapa="CONEXAO",
        objeto="DB2_PORTA",
    ) from exc

cliente_db2 = Db2(
    user=configuracao_db2["DB2_USER"],
    password=configuracao_db2["DB2_PASSWORD"],
    host=configuracao_db2["DB2_HOST"],
    port=porta_db2,
    database=configuracao_db2["DB2_DATABASE"],
    trust_env=True,
    pconnect=True,
)

print("Conexao DB2 local configurada para iniciar as consultas.")


# Extracao financeira por periodos fechados

A rotina identifica a ultima referencia temporal valida do cliente, descarta o ciclo financeiro em andamento e consulta os ultimos periodos completamente encerrados.

A execucao ocorre integralmente no kernel Python local. As tres consultas DB2 permanecem explicitas, parametrizadas e auditaveis.


In [ ]:
# Parametros editaveis.
CD_CLI = None
PERIODOS = 1

# Reinicializacao obrigatoria para evitar resultados residuais.
df_validacoes = None
df_resumo_execucao = None
df_periodos_utilizados = None
df_tabela_alvo = None
df_estudo_entradas = None
df_resumo_entradas = None
df_radar = None
validacoes = []

COLUNAS_VALIDACOES = ["ETAPA", "REGRA", "STATUS", "QT_AFETADA", "DETALHE"]


def registrar_validacao(etapa, regra, status, qt_afetada=0, detalhe=""):
    validacoes.append({
        "ETAPA": str(etapa),
        "REGRA": str(regra),
        "STATUS": str(status),
        "QT_AFETADA": int(qt_afetada),
        "DETALHE": str(detalhe),
    })


def bloquear(etapa, codigo, mensagem, qt_afetada=0, detalhe=""):
    global df_validacoes, df_tabela_alvo

    df_tabela_alvo = None
    registrar_validacao(
        etapa=etapa,
        regra=codigo,
        status="BLOQUEIO",
        qt_afetada=qt_afetada,
        detalhe=detalhe or mensagem,
    )
    df_validacoes = pd.DataFrame(validacoes, columns=COLUNAS_VALIDACOES)

    print("### VALIDACOES")
    display(df_validacoes)
    if df_resumo_execucao is not None:
        print("### RESUMO_EXECUCAO (contexto calculado)")
        display(df_resumo_execucao)
    if df_periodos_utilizados is not None:
        print("### PERIODOS_UTILIZADOS (contexto calculado)")
        display(df_periodos_utilizados.sort_values("NR_PERIODO", kind="mergesort"))

    raise ErroContratoDados(
        codigo=codigo,
        mensagem=mensagem,
        etapa=etapa,
        objeto="estudo_simplificado",
        detalhes={
            "qt_afetada": int(qt_afetada),
            "detalhe": detalhe or mensagem,
        },
        acao="Consultar VALIDACOES e corrigir a causa antes de executar novamente.",
    )


def calcular_periodos_fechados(dt_visualizacao, dia_inicio, quantidade):
    indice_mes_visualizacao = (
        dt_visualizacao.year * 12 + dt_visualizacao.month - 1
    )
    ano_mes = indice_mes_visualizacao // 12
    mes_mes = indice_mes_visualizacao % 12 + 1
    inicio_mes_visualizacao = date(
        ano_mes,
        mes_mes,
        min(dia_inicio, monthrange(ano_mes, mes_mes)[1]),
    )

    indice_mes_aberto = (
        indice_mes_visualizacao - 1
        if dt_visualizacao < inicio_mes_visualizacao
        else indice_mes_visualizacao
    )
    ano_aberto = indice_mes_aberto // 12
    mes_aberto = indice_mes_aberto % 12 + 1
    dt_inicio_aberto = date(
        ano_aberto,
        mes_aberto,
        min(dia_inicio, monthrange(ano_aberto, mes_aberto)[1]),
    )

    indice_mes_seguinte = indice_mes_aberto + 1
    ano_seguinte = indice_mes_seguinte // 12
    mes_seguinte = indice_mes_seguinte % 12 + 1
    dt_fim_exclusivo_aberto = date(
        ano_seguinte,
        mes_seguinte,
        min(dia_inicio, monthrange(ano_seguinte, mes_seguinte)[1]),
    )

    periodos_calculados = []
    dt_fim_exclusivo = dt_inicio_aberto
    for nr_periodo in range(1, quantidade + 1):
        indice_inicio = indice_mes_aberto - nr_periodo
        ano_inicio = indice_inicio // 12
        mes_inicio = indice_inicio % 12 + 1
        dt_inicio = date(
            ano_inicio,
            mes_inicio,
            min(dia_inicio, monthrange(ano_inicio, mes_inicio)[1]),
        )
        dt_fim = dt_fim_exclusivo - timedelta(days=1)
        periodos_calculados.append({
            "NR_PERIODO": nr_periodo,
            "REF_PERIODO": dt_fim.strftime("%m/%Y"),
            "DT_INICIO_PERIODO": dt_inicio,
            "DT_FIM_PERIODO": dt_fim,
            "DT_FIM_EXCLUSIVO": dt_fim_exclusivo,
        })
        dt_fim_exclusivo = dt_inicio

    return periodos_calculados, dt_inicio_aberto, dt_fim_exclusivo_aberto


def inteiro_exato(valor):
    if isinstance(valor, bool) or valor is None or pd.isna(valor):
        return None
    if isinstance(valor, Integral):
        return int(valor)
    if isinstance(valor, Decimal) and valor == valor.to_integral_value():
        return int(valor)
    return None


def decimal_exato(valor):
    if isinstance(valor, bool) or isinstance(valor, float):
        raise ValueError("bool e float nao sao valores monetarios validos")
    if isinstance(valor, Decimal):
        resultado = valor
    elif isinstance(valor, Integral):
        resultado = Decimal(int(valor))
    elif isinstance(valor, str):
        try:
            resultado = Decimal(valor.strip())
        except InvalidOperation as exc:
            raise ValueError("texto decimal invalido") from exc
    else:
        raise ValueError("tipo monetario invalido")
    if not resultado.is_finite():
        raise ValueError("decimal nao finito")
    return resultado


if CD_CLI is None:
    CD_CLI = random.randint(1, 999_999_999)
    ORIGEM_CD_CLI = "AUTOMATICO"
else:
    ORIGEM_CD_CLI = "INFORMADO"

if (
    isinstance(CD_CLI, bool)
    or not isinstance(CD_CLI, int)
    or CD_CLI < 1
    or CD_CLI > 999_999_999
):
    bloquear(
        "PARAMETROS",
        "PARAMETRO_CD_CLI_INVALIDO",
        "CD_CLI deve ser um inteiro entre 1 e 999999999.",
        detalhe=f"valor={CD_CLI!r}",
    )

if (
    isinstance(PERIODOS, bool)
    or not isinstance(PERIODOS, int)
    or PERIODOS not in (1, 2, 3, 4)
):
    bloquear(
        "PARAMETROS",
        "PARAMETRO_PERIODOS_INVALIDO",
        "PERIODOS deve ser um inteiro entre 1 e 4.",
        detalhe=f"valor={PERIODOS!r}",
    )

if ambiente == "MODELAGEM":
    HOJE = datetime.now(ZoneInfo("America/Sao_Paulo")).date()
else:
    valor_hoje = (
        os.environ.get("ctmodate") or os.environ.get("CTMODATE") or ""
    ).strip()
    try:
        HOJE = date.fromisoformat(valor_hoje)
    except ValueError:
        bloquear(
            "PARAMETROS",
            "HOJE_INVALIDO",
            "ctmodate/CTMODATE deve usar o formato AAAA-MM-DD.",
        )

registrar_validacao(
    "PARAMETROS",
    "PARAMETROS_VALIDOS",
    "OK",
    detalhe=f"CD_CLI={CD_CLI}; origem={ORIGEM_CD_CLI}; PERIODOS={PERIODOS}",
)
print(
    f"Parametros: CD_CLI={CD_CLI} ({ORIGEM_CD_CLI}); "
    f"PERIODOS={PERIODOS}; HOJE={HOJE.isoformat()}."
)


## 1. Referencia temporal

A tabela de funcionalidades e apenas a ponte entre cliente e conta GFP. As contas sao deduplicadas antes do relacionamento com `CT_GRDR_FNCO`.


In [ ]:
sql_referencia = """
SELECT
    F.CD_CLI,
    F.CD_UOR_CC,
    F.NR_CC,
    C.CD_UOR_CC AS CD_UOR_CC_CT,
    C.NR_CC AS NR_CC_CT,
    C.TS_ULT_ACSS_CT,
    C.DD_INC_MM_CLC_BLC
FROM (
    SELECT DISTINCT
        CD_CLI,
        CD_UOR_CC,
        NR_CC
    FROM DB2GFP.FUC_ACSD_GRDR_FNCO
    WHERE CD_CLI = ?
) F
LEFT JOIN DB2GFP.CT_GRDR_FNCO C
    ON C.CD_UOR_CC = F.CD_UOR_CC
   AND C.NR_CC = F.NR_CC
""".strip()

try:
    df_referencia = executar_select(sql_referencia, params=[CD_CLI])
except Exception as exc:
    bloquear(
        "REFERENCIA_TEMPORAL",
        "ERRO_TECNICO",
        "Falha ao consultar a referencia temporal.",
        detalhe=f"tipo_erro={type(exc).__name__}",
    )

colunas_referencia = {
    "CD_CLI", "CD_UOR_CC", "NR_CC", "CD_UOR_CC_CT", "NR_CC_CT",
    "TS_ULT_ACSS_CT", "DD_INC_MM_CLC_BLC",
}
if df_referencia.empty and len(df_referencia.columns) == 0:
    df_referencia = pd.DataFrame(columns=sorted(colunas_referencia))
if not colunas_referencia.issubset(df_referencia.columns):
    bloquear(
        "REFERENCIA_TEMPORAL",
        "CONTRATO_COLUNAS_INVALIDO",
        "A consulta de referencia nao retornou todas as colunas esperadas.",
    )
if df_referencia.empty:
    bloquear(
        "REFERENCIA_TEMPORAL",
        "CLIENTE_SEM_CONTA_GFP",
        "O cliente nao possui conta localizada na ponte do GFP.",
    )

mascara_com_conta = (
    df_referencia["CD_UOR_CC_CT"].notna()
    & df_referencia["NR_CC_CT"].notna()
)
df_referencias_validas = df_referencia.loc[mascara_com_conta].copy()
qt_contas_sem_correspondencia = int((~mascara_com_conta).sum())
if df_referencias_validas.empty:
    bloquear(
        "REFERENCIA_TEMPORAL",
        "CONTA_GFP_NAO_LOCALIZADA",
        "Nenhuma conta da ponte foi localizada em CT_GRDR_FNCO.",
        qt_afetada=len(df_referencia),
    )
if qt_contas_sem_correspondencia:
    registrar_validacao(
        "REFERENCIA_TEMPORAL",
        "CONTAS_PONTE_SEM_CORRESPONDENCIA",
        "INFO",
        qt_afetada=qt_contas_sem_correspondencia,
        detalhe="Existem contas antigas na ponte sem correspondencia; outras referencias validas foram encontradas.",
    )

conflitos_chave = 0
for _, grupo in df_referencias_validas.groupby(
    ["CD_UOR_CC", "NR_CC"], dropna=False, sort=False
):
    definicoes = {
        (
            None if pd.isna(linha["TS_ULT_ACSS_CT"]) else str(linha["TS_ULT_ACSS_CT"]),
            None if pd.isna(linha["DD_INC_MM_CLC_BLC"]) else str(linha["DD_INC_MM_CLC_BLC"]),
        )
        for linha in grupo.to_dict("records")
    }
    conflitos_chave += int(len(definicoes) > 1)
if conflitos_chave:
    bloquear(
        "REFERENCIA_TEMPORAL",
        "INTEGRIDADE_CHAVE_CT_GRDR_FNCO",
        "Uma chave de CT_GRDR_FNCO retornou referencias conflitantes.",
        qt_afetada=conflitos_chave,
    )

qt_ultimo_acesso_ausente = int(
    df_referencias_validas["TS_ULT_ACSS_CT"].isna().sum()
)
if qt_ultimo_acesso_ausente:
    bloquear(
        "REFERENCIA_TEMPORAL",
        "ULTIMO_ACESSO_AUSENTE",
        "TS_ULT_ACSS_CT deve estar preenchido em todas as contas localizadas.",
        qt_afetada=qt_ultimo_acesso_ausente,
    )

timestamps = pd.to_datetime(
    df_referencias_validas["TS_ULT_ACSS_CT"], errors="coerce"
)
qt_ultimo_acesso_invalido = int(timestamps.isna().sum())
if qt_ultimo_acesso_invalido:
    bloquear(
        "REFERENCIA_TEMPORAL",
        "ULTIMO_ACESSO_INVALIDO",
        "TS_ULT_ACSS_CT deve ser um timestamp valido.",
        qt_afetada=qt_ultimo_acesso_invalido,
    )
df_referencias_validas["TS_ULT_ACSS_CT"] = timestamps

cortes_normalizados = df_referencias_validas["DD_INC_MM_CLC_BLC"].map(
    inteiro_exato
)
mascara_corte_invalido = cortes_normalizados.isna() | ~cortes_normalizados.between(1, 31)
if mascara_corte_invalido.any():
    bloquear(
        "REFERENCIA_TEMPORAL",
        "DIA_INICIO_CICLO_INVALIDO",
        "DD_INC_MM_CLC_BLC deve ser um inteiro entre 1 e 31.",
        qt_afetada=int(mascara_corte_invalido.sum()),
    )
df_referencias_validas["DD_INC_MM_CLC_BLC"] = cortes_normalizados.astype(int)

mascara_futuro = df_referencias_validas["TS_ULT_ACSS_CT"].dt.date > HOJE
if mascara_futuro.any():
    bloquear(
        "REFERENCIA_TEMPORAL",
        "ULTIMO_ACESSO_NO_FUTURO",
        "TS_ULT_ACSS_CT nao pode possuir data posterior a HOJE.",
        qt_afetada=int(mascara_futuro.sum()),
        detalhe=f"HOJE={HOJE.isoformat()}",
    )

maior_ultimo_acesso = df_referencias_validas["TS_ULT_ACSS_CT"].max()
linhas_mais_recentes = df_referencias_validas.loc[
    df_referencias_validas["TS_ULT_ACSS_CT"].eq(maior_ultimo_acesso)
].copy()
cortes_mais_recentes = sorted(
    linhas_mais_recentes["DD_INC_MM_CLC_BLC"].unique().tolist()
)
if len(cortes_mais_recentes) > 1:
    bloquear(
        "REFERENCIA_TEMPORAL",
        "REFERENCIA_TEMPORAL_AMBIGUA",
        "Contas com o mesmo maior TS_ULT_ACSS_CT possuem cortes diferentes.",
        qt_afetada=len(linhas_mais_recentes),
        detalhe=f"timestamp={maior_ultimo_acesso}; cortes={cortes_mais_recentes}",
    )

linhas_mais_recentes["_ORDEM_UOR"] = linhas_mais_recentes["CD_UOR_CC"].astype(str)
linhas_mais_recentes["_ORDEM_CC"] = linhas_mais_recentes["NR_CC"].astype(str)
linha_referencia = linhas_mais_recentes.sort_values(
    ["_ORDEM_UOR", "_ORDEM_CC"], kind="mergesort"
).iloc[0]
TS_ULT_ACSS_CT = pd.Timestamp(linha_referencia["TS_ULT_ACSS_CT"]).to_pydatetime()
DT_VISUALIZACAO = TS_ULT_ACSS_CT.date()
DD_INC_MM_CLC_BLC = int(linha_referencia["DD_INC_MM_CLC_BLC"])

registrar_validacao(
    "REFERENCIA_TEMPORAL",
    "REFERENCIA_TEMPORAL_VALIDA",
    "OK",
    detalhe=(
        f"contas_candidatas={len(df_referencia)}; "
        f"timestamp={TS_ULT_ACSS_CT}; corte={DD_INC_MM_CLC_BLC}"
    ),
)


## 2. Periodos fechados

O ciclo que contem a visualizacao e aberto e nunca participa do resultado. O horizonte aceito comeca em `HOJE - 18 meses-calendario`.


In [ ]:
(
    periodos_calculados,
    DT_INICIO_PERIODO_ABERTO,
    DT_FIM_EXCLUSIVO_PERIODO_ABERTO,
) = calcular_periodos_fechados(
    DT_VISUALIZACAO,
    DD_INC_MM_CLC_BLC,
    PERIODOS,
)

periodos_inconsistentes = []
if len(periodos_calculados) != PERIODOS:
    periodos_inconsistentes.append("quantidade incorreta")
if [item["NR_PERIODO"] for item in periodos_calculados] != list(
    range(1, PERIODOS + 1)
):
    periodos_inconsistentes.append("numeracao incorreta")
if not (
    DT_INICIO_PERIODO_ABERTO
    <= DT_VISUALIZACAO
    < DT_FIM_EXCLUSIVO_PERIODO_ABERTO
):
    periodos_inconsistentes.append("visualizacao fora do ciclo aberto")

for indice, item in enumerate(periodos_calculados):
    if item["DT_INICIO_PERIODO"] > item["DT_FIM_PERIODO"]:
        periodos_inconsistentes.append(
            f"periodo {item['NR_PERIODO']} invertido"
        )
    if item["DT_FIM_PERIODO"] + timedelta(days=1) != item["DT_FIM_EXCLUSIVO"]:
        periodos_inconsistentes.append(
            f"periodo {item['NR_PERIODO']} com fronteira invalida"
        )
    if item["DT_FIM_EXCLUSIVO"] > DT_INICIO_PERIODO_ABERTO:
        periodos_inconsistentes.append(
            f"periodo {item['NR_PERIODO']} invade o ciclo aberto"
        )
    if indice > 0:
        mais_recente = periodos_calculados[indice - 1]
        if item["DT_FIM_EXCLUSIVO"] != mais_recente["DT_INICIO_PERIODO"]:
            periodos_inconsistentes.append(
                f"lacuna entre periodos {indice} e {indice + 1}"
            )

if periodos_calculados[0]["DT_FIM_EXCLUSIVO"] != DT_INICIO_PERIODO_ABERTO:
    periodos_inconsistentes.append("periodo 1 nao antecede o ciclo aberto")
if periodos_inconsistentes:
    bloquear(
        "PERIODOS",
        "CALCULO_PERIODO_INCONSISTENTE",
        "O calculo dos periodos financeiros violou o contrato.",
        qt_afetada=len(periodos_inconsistentes),
        detalhe="; ".join(periodos_inconsistentes),
    )

indice_limite_horizonte = HOJE.year * 12 + HOJE.month - 1 - 18
ano_limite_horizonte = indice_limite_horizonte // 12
mes_limite_horizonte = indice_limite_horizonte % 12 + 1
DT_LIMITE_HORIZONTE = date(
    ano_limite_horizonte,
    mes_limite_horizonte,
    min(HOJE.day, monthrange(ano_limite_horizonte, mes_limite_horizonte)[1]),
)
DT_INICIO_JANELA = min(
    item["DT_INICIO_PERIODO"] for item in periodos_calculados
)
DT_FIM_EXCLUSIVO_JANELA = DT_INICIO_PERIODO_ABERTO
if DT_INICIO_JANELA < DT_LIMITE_HORIZONTE:
    bloquear(
        "PERIODOS",
        "PERIODO_FORA_HORIZONTE_TABELA_PRINCIPAL",
        "O periodo mais antigo ultrapassa os 18 meses suportados pela tabela principal.",
        qt_afetada=1,
        detalhe=(
            f"inicio_solicitado={DT_INICIO_JANELA.isoformat()}; "
            f"limite={DT_LIMITE_HORIZONTE.isoformat()}"
        ),
    )

linhas_periodos = [
    {
        "CD_CLI": CD_CLI,
        **item,
    }
    for item in periodos_calculados
]
df_periodos_interno = pd.DataFrame(linhas_periodos)
df_periodos_utilizados = df_periodos_interno.drop(
    columns=["DT_FIM_EXCLUSIVO"]
).copy()
df_resumo_execucao = pd.DataFrame([{
    "CD_CLI": CD_CLI,
    "ORIGEM_CD_CLI": ORIGEM_CD_CLI,
    "PERIODOS_SOLICITADOS": PERIODOS,
    "TS_ULT_ACSS_CT": TS_ULT_ACSS_CT,
    "DT_VISUALIZACAO": DT_VISUALIZACAO,
    "DD_INC_MM_CLC_BLC": DD_INC_MM_CLC_BLC,
    "QT_PERIODOS_CALCULADOS": len(periodos_calculados),
    "STATUS_VALIDACAO": "EM_VALIDACAO",
}])

# Casos locais de fronteira da regra temporal.
for corte_teste in (1, 29, 30, 31):
    for acesso_teste in (
        date(2024, 2, 1),
        date(2024, 2, 29),
        date(2025, 2, 28),
        date(2025, 12, 31),
        date(2026, 1, 1),
    ):
        for quantidade_teste in (1, 2, 3, 4):
            calculados_teste, inicio_aberto_teste, fim_aberto_teste = (
                calcular_periodos_fechados(
                    acesso_teste,
                    corte_teste,
                    quantidade_teste,
                )
            )
            assert len(calculados_teste) == quantidade_teste
            assert inicio_aberto_teste <= acesso_teste < fim_aberto_teste
            assert calculados_teste[0]["DT_FIM_EXCLUSIVO"] == inicio_aberto_teste
assert DT_INICIO_JANELA == DT_LIMITE_HORIZONTE or (
    DT_INICIO_JANELA > DT_LIMITE_HORIZONTE
)

registrar_validacao(
    "PERIODOS",
    "PERIODOS_VALIDOS",
    "OK",
    detalhe=(
        f"janela=[{DT_INICIO_JANELA.isoformat()}, "
        f"{DT_FIM_EXCLUSIVO_JANELA.isoformat()}); "
        f"limite_horizonte={DT_LIMITE_HORIZONTE.isoformat()}"
    ),
)


## 3. Dominio padrao de grupos e categorias

O `LEFT JOIN` preserva categorias cujo grupo esteja ausente, permitindo diagnosticar `GRUPO_NAO_MAPEADO` separadamente.


In [ ]:
sql_categorias = """
SELECT
    C.CD_CTGR_TRAN,
    C.CD_GR_CTGR_TRAN,
    C.CD_NTZ_CTB_TRAN,
    RTRIM(C.TX_DCR_CTGR_TRAN) AS TX_DCR_CTGR_TRAN,
    RTRIM(G.TX_DCR_GR_CTGR) AS TX_DCR_GR_CTGR
FROM DB2GFP.CTGR_TRAN_OPB C
LEFT JOIN DB2GFP.GR_CTGR_TRAN G
    ON G.CD_GR_CTGR_TRAN = C.CD_GR_CTGR_TRAN
WHERE TRIM(C.IN_CTGR_PDRO_SIS) = 'S'
""".strip()

try:
    df_categorias_bruto = executar_select(sql_categorias)
except Exception as exc:
    bloquear(
        "DOMINIO_CATEGORIAS",
        "ERRO_TECNICO",
        "Falha ao consultar o dominio de grupos e categorias.",
        detalhe=f"tipo_erro={type(exc).__name__}",
    )

colunas_categorias = {
    "CD_CTGR_TRAN", "CD_GR_CTGR_TRAN", "CD_NTZ_CTB_TRAN",
    "TX_DCR_CTGR_TRAN", "TX_DCR_GR_CTGR",
}
if df_categorias_bruto.empty and len(df_categorias_bruto.columns) == 0:
    df_categorias_bruto = pd.DataFrame(columns=sorted(colunas_categorias))
if not colunas_categorias.issubset(df_categorias_bruto.columns):
    bloquear(
        "DOMINIO_CATEGORIAS",
        "CONTRATO_COLUNAS_INVALIDO",
        "A consulta de categorias nao retornou todas as colunas esperadas.",
    )
if df_categorias_bruto.empty:
    bloquear(
        "DOMINIO_CATEGORIAS",
        "DICIONARIO_CATEGORIA_INCONSISTENTE",
        "O dominio padrao de categorias esta vazio.",
    )

df_categorias_bruto["CD_NTZ_CTB_CTGR"] = (
    df_categorias_bruto["CD_NTZ_CTB_TRAN"]
    .astype("string")
    .str.strip()
    .str.upper()
    .fillna("")
)
conflitos_categoria = 0
for _, grupo in df_categorias_bruto.groupby(
    "CD_CTGR_TRAN", dropna=False, sort=False
):
    definicoes = {
        (
            None if pd.isna(linha["CD_GR_CTGR_TRAN"]) else str(linha["CD_GR_CTGR_TRAN"]),
            linha["CD_NTZ_CTB_CTGR"],
            None if pd.isna(linha["TX_DCR_CTGR_TRAN"]) else str(linha["TX_DCR_CTGR_TRAN"]),
        )
        for linha in grupo.to_dict("records")
    }
    conflitos_categoria += int(len(definicoes) > 1)
if conflitos_categoria:
    bloquear(
        "DOMINIO_CATEGORIAS",
        "DICIONARIO_CATEGORIA_INCONSISTENTE",
        "Uma categoria padrao possui definicoes conflitantes.",
        qt_afetada=conflitos_categoria,
    )

conflitos_grupo = 0
grupos_preenchidos = df_categorias_bruto.loc[
    df_categorias_bruto["CD_GR_CTGR_TRAN"].notna()
]
for _, grupo in grupos_preenchidos.groupby(
    "CD_GR_CTGR_TRAN", dropna=False, sort=False
):
    descricoes = {
        "__AUSENTE__" if pd.isna(valor) else str(valor)
        for valor in grupo["TX_DCR_GR_CTGR"]
    }
    conflitos_grupo += int(len(descricoes) > 1)
if conflitos_grupo:
    bloquear(
        "DOMINIO_CATEGORIAS",
        "DICIONARIO_GRUPO_INCONSISTENTE",
        "Um grupo possui descricoes conflitantes no dominio.",
        qt_afetada=conflitos_grupo,
    )

mascara_natureza_invalida = ~df_categorias_bruto[
    "CD_NTZ_CTB_CTGR"
].isin(["", "C", "D"])
if mascara_natureza_invalida.any():
    bloquear(
        "DOMINIO_CATEGORIAS",
        "DICIONARIO_CATEGORIA_INCONSISTENTE",
        "O dominio possui natureza diferente de vazio, C ou D.",
        qt_afetada=int(mascara_natureza_invalida.sum()),
    )

df_categorias = (
    df_categorias_bruto
    .drop_duplicates(subset=["CD_CTGR_TRAN"], keep="first")
    [[
        "CD_CTGR_TRAN", "CD_GR_CTGR_TRAN", "CD_NTZ_CTB_CTGR",
        "TX_DCR_CTGR_TRAN", "TX_DCR_GR_CTGR",
    ]]
    .copy()
)
registrar_validacao(
    "DOMINIO_CATEGORIAS",
    "DICIONARIO_VALIDO",
    "OK",
    detalhe=f"categorias_padrao={len(df_categorias)}",
)


## 4. Transacoes

A tabela transacional e filtrada no DB2 por cliente e pela janela semiaberta dos periodos fechados. Nenhum estado e removido antecipadamente.


In [ ]:
sql_transacoes = """
SELECT
    NR_TRAN_INST_PCT,
    CD_CLI,
    NR_MCA_PCT_OPB,
    DT_TRAN,
    CD_NTZ_CTB_TRAN,
    CD_EST_TRAN_INST,
    CD_CTGR_TRAN,
    CD_TIP_MOE_CRR,
    VL_TRAN,
    TX_DCR_TRAN
FROM DB2GFP.TRAN_RLZD_INST_PCT
WHERE CD_CLI = ?
  AND DT_TRAN >= DATE(?)
  AND DT_TRAN < DATE(?)
""".strip()

try:
    df_transacoes = executar_select(
        sql_transacoes,
        params=[
            CD_CLI,
            DT_INICIO_JANELA.isoformat(),
            DT_FIM_EXCLUSIVO_JANELA.isoformat(),
        ],
        chunksize=DB2_CHUNKSIZE,
    )
    QT_TRANSACOES_ORIGINAIS = int(len(df_transacoes))
except Exception as exc:
    detalhes_volume = dict(ultima_leitura_db2)
    detalhes_volume.update({
        "CD_CLI": CD_CLI,
        "inicio": DT_INICIO_JANELA.isoformat(),
        "fim_exclusivo": DT_FIM_EXCLUSIVO_JANELA.isoformat(),
    })
    codigo_extracao = (
        exc.codigo
        if isinstance(exc, ErroContratoDados)
        and exc.codigo == "VOLUME_LOCAL_INCOMPATIVEL"
        else "ERRO_TECNICO"
    )
    bloquear(
        "TRANSACOES",
        codigo_extracao,
        "Falha ao extrair as transacoes para Pandas.",
        detalhe=f"tipo_erro={type(exc).__name__}; contexto={detalhes_volume}",
    )

colunas_transacoes = [
    "NR_TRAN_INST_PCT", "CD_CLI", "NR_MCA_PCT_OPB", "DT_TRAN",
    "CD_NTZ_CTB_TRAN", "CD_EST_TRAN_INST", "CD_CTGR_TRAN",
    "CD_TIP_MOE_CRR", "VL_TRAN", "TX_DCR_TRAN",
]
if df_transacoes.empty and len(df_transacoes.columns) == 0:
    df_transacoes = pd.DataFrame(columns=colunas_transacoes)
if not set(colunas_transacoes).issubset(df_transacoes.columns):
    bloquear(
        "TRANSACOES",
        "CONTRATO_COLUNAS_INVALIDO",
        "A consulta transacional nao retornou as dez colunas esperadas.",
    )
df_transacoes = df_transacoes[colunas_transacoes].copy()

registrar_validacao(
    "TRANSACOES",
    "EXTRACAO_COMPLETA",
    "OK",
    detalhe=(
        f"linhas={QT_TRANSACOES_ORIGINAIS}; "
        f"chunks={ultima_leitura_db2['chunks']}; "
        f"memoria_estimada_bytes={ultima_leitura_db2['memoria_estimada_bytes']}"
    ),
)

mascara_outro_cliente = (
    df_transacoes["CD_CLI"].isna()
    | ~df_transacoes["CD_CLI"].eq(CD_CLI)
)
if mascara_outro_cliente.any():
    bloquear(
        "TRANSACOES",
        "VAZAMENTO_ENTRE_CLIENTES",
        "A extracao retornou transacoes de outro cliente.",
        qt_afetada=int(mascara_outro_cliente.sum()),
    )

qt_chaves_ausentes = int(df_transacoes["NR_TRAN_INST_PCT"].isna().sum())
qt_chaves_duplicadas = int(
    df_transacoes.loc[
        df_transacoes["NR_TRAN_INST_PCT"].duplicated(keep=False),
        "NR_TRAN_INST_PCT",
    ].nunique(dropna=True)
)
if qt_chaves_ausentes or qt_chaves_duplicadas:
    bloquear(
        "TRANSACOES",
        "TRANSACAO_DUPLICADA_NA_FONTE",
        "NR_TRAN_INST_PCT deve estar preenchido e ser unico no recorte.",
        qt_afetada=qt_chaves_ausentes + qt_chaves_duplicadas,
    )

datas_convertidas = pd.to_datetime(df_transacoes["DT_TRAN"], errors="coerce")
datas_normalizadas = datas_convertidas.dt.date
mascara_data_invalida = (
    datas_convertidas.isna()
    | datas_normalizadas.lt(DT_INICIO_JANELA)
    | datas_normalizadas.ge(DT_FIM_EXCLUSIVO_JANELA)
)
if mascara_data_invalida.any():
    bloquear(
        "TRANSACOES",
        "TRANSACAO_FORA_DOS_PERIODOS",
        "A extracao contem DT_TRAN invalida ou fora da janela.",
        qt_afetada=int(mascara_data_invalida.sum()),
    )
df_transacoes["DT_TRAN"] = datas_normalizadas

df_transacoes["CD_NTZ_CTB_TRAN"] = (
    df_transacoes["CD_NTZ_CTB_TRAN"]
    .astype("string").str.strip().str.upper()
)
mascara_natureza_invalida = (
    df_transacoes["CD_NTZ_CTB_TRAN"].isna()
    | ~df_transacoes["CD_NTZ_CTB_TRAN"].isin(["C", "D"])
)
if mascara_natureza_invalida.any():
    bloquear(
        "TRANSACOES",
        "NATUREZA_CONTABIL_INVALIDA",
        "CD_NTZ_CTB_TRAN deve ser C ou D.",
        qt_afetada=int(mascara_natureza_invalida.sum()),
    )

estados_normalizados = df_transacoes["CD_EST_TRAN_INST"].map(inteiro_exato)
mascara_estado_invalido = estados_normalizados.isna() | ~estados_normalizados.isin([0, 1, 9])
if mascara_estado_invalido.any():
    bloquear(
        "TRANSACOES",
        "ESTADO_TRANSACAO_INVALIDO",
        "CD_EST_TRAN_INST deve ser 0, 1 ou 9.",
        qt_afetada=int(mascara_estado_invalido.sum()),
    )
df_transacoes["CD_EST_TRAN_INST"] = estados_normalizados.astype(int)

qt_contas_ausentes = int(df_transacoes["NR_MCA_PCT_OPB"].isna().sum())
if qt_contas_ausentes:
    bloquear(
        "TRANSACOES",
        "CONTA_IDENTIFICADA_AUSENTE",
        "NR_MCA_PCT_OPB deve estar preenchido.",
        qt_afetada=qt_contas_ausentes,
    )

df_transacoes["CD_TIP_MOE_CRR"] = (
    df_transacoes["CD_TIP_MOE_CRR"].astype("string").str.strip().str.upper()
)
mascara_moeda_ausente = (
    df_transacoes["CD_TIP_MOE_CRR"].isna()
    | df_transacoes["CD_TIP_MOE_CRR"].eq("")
)
if mascara_moeda_ausente.any():
    bloquear(
        "TRANSACOES",
        "MOEDA_AUSENTE",
        "CD_TIP_MOE_CRR deve estar preenchido.",
        qt_afetada=int(mascara_moeda_ausente.sum()),
    )

valores_normalizados = []
indices_valor_invalido = []
for indice, valor in df_transacoes["VL_TRAN"].items():
    try:
        valor_decimal = decimal_exato(valor)
        if valor_decimal < 0:
            raise ValueError("valor negativo")
        valores_normalizados.append(valor_decimal)
    except ValueError:
        indices_valor_invalido.append(indice)
        valores_normalizados.append(None)
if indices_valor_invalido:
    bloquear(
        "TRANSACOES",
        "VALOR_TRANSACAO_INVALIDO",
        "VL_TRAN deve ser decimal exato, preenchido e nao negativo.",
        qt_afetada=len(indices_valor_invalido),
    )
df_transacoes["VL_TRAN"] = valores_normalizados

mascara_descricao_ausente = (
    df_transacoes["TX_DCR_TRAN"].isna()
    | df_transacoes["TX_DCR_TRAN"].astype("string").str.strip().eq("")
)
if mascara_descricao_ausente.any():
    bloquear(
        "TRANSACOES",
        "DESCRICAO_TRANSACAO_AUSENTE",
        "TX_DCR_TRAN deve estar preenchida.",
        qt_afetada=int(mascara_descricao_ausente.sum()),
    )
df_transacoes["TX_DCR_TRAN"] = (
    df_transacoes["TX_DCR_TRAN"].astype("string").str.rstrip()
)

descricoes_estado = {
    0: "TRANSACAO_EFETIVADA",
    1: "LANCAMENTO_FUTURO",
    9: "DELECAO_LOGICA",
}
df_transacoes_estado = df_transacoes.copy()
df_transacoes_estado["TX_EST_TRAN_INST"] = (
    df_transacoes_estado["CD_EST_TRAN_INST"].map(descricoes_estado)
)
registrar_validacao(
    "TRANSACOES",
    "TRANSACOES_VALIDAS",
    "OK",
    detalhe=f"transacoes={QT_TRANSACOES_ORIGINAIS}",
)


## 5. Atribuicao de periodo e enriquecimento

Cada transacao deve pertencer a exatamente um periodo fechado. O enriquecimento preserva o grao tecnico original.


In [ ]:
df_transacoes_periodos = df_transacoes_estado.copy()
df_transacoes_periodos["NR_PERIODO"] = pd.Series(
    [pd.NA] * len(df_transacoes_periodos), dtype="Int64"
)
df_transacoes_periodos["REF_PERIODO"] = pd.Series(
    [pd.NA] * len(df_transacoes_periodos), dtype="string"
)
df_transacoes_periodos["DT_INICIO_PERIODO"] = None
df_transacoes_periodos["DT_FIM_PERIODO"] = None
df_transacoes_periodos["DT_FIM_EXCLUSIVO"] = None
qt_associacoes = pd.Series(0, index=df_transacoes_periodos.index, dtype="int64")

for periodo in periodos_calculados:
    mascara = (
        df_transacoes_periodos["DT_TRAN"].ge(periodo["DT_INICIO_PERIODO"])
        & df_transacoes_periodos["DT_TRAN"].lt(periodo["DT_FIM_EXCLUSIVO"])
    )
    qt_associacoes.loc[mascara] += 1
    df_transacoes_periodos.loc[mascara, "NR_PERIODO"] = periodo["NR_PERIODO"]
    df_transacoes_periodos.loc[mascara, "REF_PERIODO"] = periodo["REF_PERIODO"]
    for coluna in (
        "DT_INICIO_PERIODO", "DT_FIM_PERIODO", "DT_FIM_EXCLUSIVO"
    ):
        df_transacoes_periodos.loc[mascara, coluna] = periodo[coluna]

qt_sem_periodo = int(qt_associacoes.eq(0).sum())
qt_multiplos_periodos = int(qt_associacoes.gt(1).sum())
if qt_sem_periodo:
    bloquear(
        "ATRIBUICAO_PERIODO",
        "TRANSACAO_FORA_DOS_PERIODOS",
        "Uma transacao nao foi associada a nenhum periodo.",
        qt_afetada=qt_sem_periodo,
    )
if qt_multiplos_periodos:
    bloquear(
        "ATRIBUICAO_PERIODO",
        "TRANSACAO_EM_MULTIPLOS_PERIODOS",
        "Uma transacao foi associada a mais de um periodo.",
        qt_afetada=qt_multiplos_periodos,
    )

mascara_periodo_aberto = (
    df_transacoes_periodos["DT_TRAN"].ge(DT_INICIO_PERIODO_ABERTO)
    & df_transacoes_periodos["DT_TRAN"].lt(DT_FIM_EXCLUSIVO_PERIODO_ABERTO)
)
if mascara_periodo_aberto.any():
    bloquear(
        "ATRIBUICAO_PERIODO",
        "TRANSACAO_DO_PERIODO_ABERTO_INCLUIDA",
        "O resultado incluiu transacao do periodo aberto.",
        qt_afetada=int(mascara_periodo_aberto.sum()),
    )
registrar_validacao(
    "ATRIBUICAO_PERIODO",
    "ATRIBUICAO_UNICA",
    "OK",
    detalhe="Cada transacao pertence a exatamente um periodo fechado.",
)

df_transacoes_periodos["_CHAVE_CTGR_DOMINIO"] = (
    df_transacoes_periodos["CD_CTGR_TRAN"]
    .map(inteiro_exato)
    .astype("Int64")
)
df_categorias_join = df_categorias.rename(
    columns={"CD_CTGR_TRAN": "CD_CTGR_TRAN_DOMINIO"}
)
df_categorias_join["_CHAVE_CTGR_DOMINIO"] = (
    df_categorias_join["CD_CTGR_TRAN_DOMINIO"]
    .map(inteiro_exato)
    .astype("Int64")
)
try:
    df_enriquecido = df_transacoes_periodos.merge(
        df_categorias_join,
        how="left",
        on="_CHAVE_CTGR_DOMINIO",
        validate="many_to_one",
        indicator="_COBERTURA_CATEGORIA",
        sort=False,
    )
except pd.errors.MergeError as exc:
    bloquear(
        "ENRIQUECIMENTO",
        "DUPLICACAO_POR_ENRIQUECIMENTO",
        "O dominio de categorias multiplicaria o grao transacional.",
        detalhe=f"tipo_erro={type(exc).__name__}",
    )

QT_TRANSACOES_ENRIQUECIDAS = int(len(df_enriquecido))
QT_IDS_ORIGINAIS = int(df_transacoes["NR_TRAN_INST_PCT"].nunique())
QT_IDS_ENRIQUECIDOS = int(df_enriquecido["NR_TRAN_INST_PCT"].nunique())
if (
    QT_TRANSACOES_ENRIQUECIDAS > QT_TRANSACOES_ORIGINAIS
    or QT_IDS_ENRIQUECIDOS > QT_IDS_ORIGINAIS
):
    bloquear(
        "ENRIQUECIMENTO",
        "DUPLICACAO_POR_ENRIQUECIMENTO",
        "O enriquecimento multiplicou o grao transacional.",
        qt_afetada=QT_TRANSACOES_ENRIQUECIDAS - QT_TRANSACOES_ORIGINAIS,
    )
if (
    QT_TRANSACOES_ENRIQUECIDAS < QT_TRANSACOES_ORIGINAIS
    or QT_IDS_ENRIQUECIDOS < QT_IDS_ORIGINAIS
):
    bloquear(
        "ENRIQUECIMENTO",
        "PERDA_POR_ENRIQUECIMENTO",
        "O enriquecimento perdeu transacoes.",
        qt_afetada=QT_TRANSACOES_ORIGINAIS - QT_TRANSACOES_ENRIQUECIDAS,
    )

mascara_categoria_ausente = df_enriquecido["_COBERTURA_CATEGORIA"].ne("both")
if mascara_categoria_ausente.any():
    bloquear(
        "ENRIQUECIMENTO",
        "CATEGORIA_NAO_MAPEADA_NO_DOMINIO_PADRAO",
        "Uma transacao nao encontrou categoria padrao.",
        qt_afetada=int(mascara_categoria_ausente.sum()),
    )
mascara_grupo_ausente = (
    df_enriquecido["CD_CTGR_TRAN_DOMINIO"].notna()
    & (
        df_enriquecido["CD_GR_CTGR_TRAN"].isna()
        | df_enriquecido["TX_DCR_GR_CTGR"].isna()
    )
)
if mascara_grupo_ausente.any():
    bloquear(
        "ENRIQUECIMENTO",
        "GRUPO_NAO_MAPEADO",
        "Uma categoria utilizada nao encontrou grupo.",
        qt_afetada=int(mascara_grupo_ausente.sum()),
    )
mascara_natureza_incompativel = (
    df_enriquecido["CD_NTZ_CTB_CTGR"].isin(["C", "D"])
    & df_enriquecido["CD_NTZ_CTB_CTGR"].ne(
        df_enriquecido["CD_NTZ_CTB_TRAN"]
    )
)
if mascara_natureza_incompativel.any():
    bloquear(
        "ENRIQUECIMENTO",
        "INCOMPATIBILIDADE_NATUREZA_CATEGORIA",
        "A natureza da categoria diverge da natureza da transacao.",
        qt_afetada=int(mascara_natureza_incompativel.sum()),
    )

COLUNAS_TABELA_ALVO = [
    "CD_CLI", "NR_PERIODO", "REF_PERIODO", "NR_MCA_PCT_OPB",
    "DT_TRAN", "CD_NTZ_CTB_TRAN", "CD_EST_TRAN_INST",
    "TX_EST_TRAN_INST", "CD_GR_CTGR_TRAN", "TX_DCR_GR_CTGR",
    "CD_CTGR_TRAN", "TX_DCR_CTGR_TRAN", "CD_TIP_MOE_CRR",
    "VL_TRAN", "TX_DCR_TRAN",
]
df_tabela_alvo = (
    df_enriquecido[COLUNAS_TABELA_ALVO]
    .sort_values(
        ["NR_PERIODO", "DT_TRAN", "NR_MCA_PCT_OPB"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
registrar_validacao(
    "ENRIQUECIMENTO",
    "GRAO_E_COBERTURA_VALIDOS",
    "OK",
    detalhe=f"linhas={QT_TRANSACOES_ORIGINAIS}; ids={QT_IDS_ORIGINAIS}",
)


## 6. Resultado

As quatro saidas contratuais da extracao sao exibidas somente depois de todas as validacoes da camada financeira principal.


In [ ]:
registrar_validacao(
    "RESULTADO",
    "RESULTADO_LIBERADO",
    "OK",
    detalhe=f"transacoes={QT_TRANSACOES_ORIGINAIS}",
)
df_validacoes = pd.DataFrame(validacoes, columns=COLUNAS_VALIDACOES)
df_resumo_execucao = df_resumo_execucao.copy()
df_resumo_execucao["STATUS_VALIDACAO"] = "OK"

print("### VALIDACOES")
display(df_validacoes)
print("### RESUMO_EXECUCAO")
display(df_resumo_execucao)
print("### PERIODOS_UTILIZADOS")
display(df_periodos_utilizados.sort_values("NR_PERIODO", kind="mergesort"))
print("### TABELA_ALVO")
display(df_tabela_alvo)


## 7. Estudo das entradas

Creditos efetivados sao comparados a debitos efetivados do mesmo cliente por valor exato, moeda, fontes diferentes e distancia de ate tres dias.


In [ ]:
# Reinicializacao obrigatoria da camada de entradas.
df_estudo_entradas = None
df_resumo_entradas = None


def parear_movimentacoes(creditos, debitos):
    creditos_ordenados = sorted(
        [dict(item) for item in creditos],
        key=lambda item: int(item["NR_TRAN_INST_PCT"]),
    )
    debitos_ordenados = sorted(
        [dict(item) for item in debitos],
        key=lambda item: int(item["NR_TRAN_INST_PCT"]),
    )

    creditos_utilizados = set()
    debitos_utilizados = set()
    resultados = {
        credito["NR_TRAN_INST_PCT"]: {
            "NR_TRAN_CREDITO": credito["NR_TRAN_INST_PCT"],
            "FL_SANEADO": "N",
            "NIVEL_EVIDENCIA": None,
            "QT_CANDIDATOS": 0,
            "NR_TRAN_DEBITO_SELECIONADO": None,
            "NR_MCA_DEBITO_SELECIONADO": None,
            "DT_DEBITO_SELECIONADO": None,
            "VL_DEBITO_SELECIONADO": None,
            "DIFERENCA_DIAS": None,
            "MOTIVO_CLASSIFICACAO": "SEM_DEBITO_DISPONIVEL",
        }
        for credito in creditos_ordenados
    }

    for diferenca_dias in (0, 1, 2, 3):
        for credito in creditos_ordenados:
            nr_credito = credito["NR_TRAN_INST_PCT"]
            if nr_credito in creditos_utilizados:
                continue

            candidatos = []
            for debito in debitos_ordenados:
                nr_debito = debito["NR_TRAN_INST_PCT"]
                if nr_debito in debitos_utilizados:
                    continue
                diferenca_observada = abs(
                    (credito["DT_TRAN"] - debito["DT_TRAN"]).days
                )
                if (
                    str(credito["CD_NTZ_CTB_TRAN"]).strip().upper() == "C"
                    and int(credito["CD_EST_TRAN_INST"]) == 0
                    and str(debito["CD_NTZ_CTB_TRAN"]).strip().upper() == "D"
                    and int(debito["CD_EST_TRAN_INST"]) == 0
                    and credito["CD_CLI"] == debito["CD_CLI"]
                    and credito["VL_TRAN"] == debito["VL_TRAN"]
                    and str(credito["CD_TIP_MOE_CRR"]).strip()
                    == str(debito["CD_TIP_MOE_CRR"]).strip()
                    and credito["NR_MCA_PCT_OPB"]
                    != debito["NR_MCA_PCT_OPB"]
                    and diferenca_observada == diferenca_dias
                ):
                    candidatos.append(debito)

            candidatos.sort(
                key=lambda item: int(item["NR_TRAN_INST_PCT"])
            )
            if not candidatos:
                continue

            debito_escolhido = candidatos[0]
            nr_debito = debito_escolhido["NR_TRAN_INST_PCT"]
            creditos_utilizados.add(nr_credito)
            debitos_utilizados.add(nr_debito)

            if diferenca_dias == 0:
                nivel_evidencia = "MUITO_FORTE"
                motivo = "MATCH_MESMO_DIA"
            else:
                nivel_evidencia = "FORTE"
                motivo = {
                    1: "MATCH_1_DIA",
                    2: "MATCH_2_DIAS",
                    3: "MATCH_3_DIAS",
                }[diferenca_dias]

            resultados[nr_credito] = {
                "NR_TRAN_CREDITO": nr_credito,
                "FL_SANEADO": "S",
                "NIVEL_EVIDENCIA": nivel_evidencia,
                "QT_CANDIDATOS": len(candidatos),
                "NR_TRAN_DEBITO_SELECIONADO": nr_debito,
                "NR_MCA_DEBITO_SELECIONADO": debito_escolhido[
                    "NR_MCA_PCT_OPB"
                ],
                "DT_DEBITO_SELECIONADO": debito_escolhido["DT_TRAN"],
                "VL_DEBITO_SELECIONADO": debito_escolhido["VL_TRAN"],
                "DIFERENCA_DIAS": diferenca_dias,
                "MOTIVO_CLASSIFICACAO": motivo,
            }

    return [
        resultados[credito["NR_TRAN_INST_PCT"]]
        for credito in creditos_ordenados
    ]


In [ ]:
try:
    creditos_teste = [
        {"NR_TRAN_INST_PCT": 101, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 10, "DT_TRAN": date(2026, 1, 11), "CD_NTZ_CTB_TRAN": "C", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("100.00")},
        {"NR_TRAN_INST_PCT": 102, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 20, "DT_TRAN": date(2026, 1, 10), "CD_NTZ_CTB_TRAN": "C", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("100.00")},
        {"NR_TRAN_INST_PCT": 103, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 40, "DT_TRAN": date(2026, 1, 10), "CD_NTZ_CTB_TRAN": "C", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("200.00")},
        {"NR_TRAN_INST_PCT": 104, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 60, "DT_TRAN": date(2026, 1, 10), "CD_NTZ_CTB_TRAN": "C", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("300.00")},
        {"NR_TRAN_INST_PCT": 105, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 90, "DT_TRAN": date(2026, 1, 10), "CD_NTZ_CTB_TRAN": "C", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("400.00")},
        {"NR_TRAN_INST_PCT": 106, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 100, "DT_TRAN": date(2026, 1, 10), "CD_NTZ_CTB_TRAN": "C", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("500.00")},
        {"NR_TRAN_INST_PCT": 107, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 110, "DT_TRAN": date(2026, 1, 20), "CD_NTZ_CTB_TRAN": "C", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("600.00")},
        {"NR_TRAN_INST_PCT": 108, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 120, "DT_TRAN": date(2026, 1, 20), "CD_NTZ_CTB_TRAN": "C", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("600.00")},
    ]
    debitos_teste = [
        {"NR_TRAN_INST_PCT": 201, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 30, "DT_TRAN": date(2026, 1, 10), "CD_NTZ_CTB_TRAN": "D", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("100.00")},
        {"NR_TRAN_INST_PCT": 202, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 50, "DT_TRAN": date(2026, 1, 11), "CD_NTZ_CTB_TRAN": "D", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("200.00")},
        {"NR_TRAN_INST_PCT": 203, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 70, "DT_TRAN": date(2026, 1, 12), "CD_NTZ_CTB_TRAN": "D", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("300.00")},
        {"NR_TRAN_INST_PCT": 204, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 80, "DT_TRAN": date(2026, 1, 12), "CD_NTZ_CTB_TRAN": "D", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("300.00")},
        {"NR_TRAN_INST_PCT": 205, "CD_CLI": 1, "NR_PERIODO": 2, "NR_MCA_PCT_OPB": 91, "DT_TRAN": date(2026, 1, 13), "CD_NTZ_CTB_TRAN": "D", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("400.00")},
        {"NR_TRAN_INST_PCT": 206, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 100, "DT_TRAN": date(2026, 1, 10), "CD_NTZ_CTB_TRAN": "D", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("500.00")},
        {"NR_TRAN_INST_PCT": 207, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 101, "DT_TRAN": date(2026, 1, 10), "CD_NTZ_CTB_TRAN": "D", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "USD", "VL_TRAN": Decimal("500.00")},
        {"NR_TRAN_INST_PCT": 208, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 101, "DT_TRAN": date(2026, 1, 10), "CD_NTZ_CTB_TRAN": "D", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("501.00")},
        {"NR_TRAN_INST_PCT": 209, "CD_CLI": 1, "NR_PERIODO": 1, "NR_MCA_PCT_OPB": 130, "DT_TRAN": date(2026, 1, 20), "CD_NTZ_CTB_TRAN": "D", "CD_EST_TRAN_INST": 0, "CD_TIP_MOE_CRR": "BRL", "VL_TRAN": Decimal("600.00")},
    ]

    resultado_teste = parear_movimentacoes(creditos_teste, debitos_teste)
    por_credito = {
        item["NR_TRAN_CREDITO"]: item for item in resultado_teste
    }
    assert por_credito[102]["NR_TRAN_DEBITO_SELECIONADO"] == 201
    assert por_credito[102]["NIVEL_EVIDENCIA"] == "MUITO_FORTE"
    assert por_credito[101]["FL_SANEADO"] == "N"
    assert por_credito[103]["DIFERENCA_DIAS"] == 1
    assert por_credito[104]["DIFERENCA_DIAS"] == 2
    assert por_credito[104]["QT_CANDIDATOS"] == 2
    assert por_credito[104]["NR_TRAN_DEBITO_SELECIONADO"] == 203
    assert por_credito[105]["DIFERENCA_DIAS"] == 3
    assert por_credito[106]["MOTIVO_CLASSIFICACAO"] == "SEM_DEBITO_DISPONIVEL"
    assert por_credito[107]["NR_TRAN_DEBITO_SELECIONADO"] == 209
    assert por_credito[108]["FL_SANEADO"] == "N"
    ids_debito = [
        item["NR_TRAN_DEBITO_SELECIONADO"]
        for item in resultado_teste
        if item["FL_SANEADO"] == "S"
    ]
    assert len(ids_debito) == len(set(ids_debito))
    assert resultado_teste == parear_movimentacoes(creditos_teste, debitos_teste)
    assert parear_movimentacoes([], debitos_teste) == []
    try:
        decimal_exato(1.25)
        raise AssertionError("float deveria ser rejeitado")
    except ValueError:
        pass
    print("Testes sinteticos do pareamento e do decimal exato: OK")
except Exception as exc:
    df_estudo_entradas = None
    df_resumo_entradas = None
    raise ErroContratoDados(
        codigo="TESTE_PAREAMENTO_ENTRADAS_FALHOU",
        mensagem="Uma regra sintetica do pareamento de entradas falhou.",
        etapa="ESTUDO_ENTRADAS",
        objeto="parear_movimentacoes",
        detalhes={"tipo_erro": type(exc).__name__},
    ) from exc


In [ ]:
try:
    df_creditos_efetivados = df_enriquecido.loc[
        df_enriquecido["CD_NTZ_CTB_TRAN"].eq("C")
        & df_enriquecido["CD_EST_TRAN_INST"].eq(0)
    ].copy()
    df_debitos_efetivados = df_enriquecido.loc[
        df_enriquecido["CD_NTZ_CTB_TRAN"].eq("D")
        & df_enriquecido["CD_EST_TRAN_INST"].eq(0)
    ].copy()

    colunas_pareamento = [
        "NR_TRAN_INST_PCT", "CD_CLI", "NR_PERIODO", "NR_MCA_PCT_OPB",
        "DT_TRAN", "CD_NTZ_CTB_TRAN", "CD_EST_TRAN_INST",
        "CD_TIP_MOE_CRR", "VL_TRAN",
    ]
    creditos_locais = (
        df_creditos_efetivados[colunas_pareamento]
        .sort_values("NR_TRAN_INST_PCT", kind="mergesort")
        .to_dict("records")
    )
    debitos_locais = (
        df_debitos_efetivados[colunas_pareamento]
        .sort_values("NR_TRAN_INST_PCT", kind="mergesort")
        .to_dict("records")
    )
    pareamentos_entradas = parear_movimentacoes(
        creditos_locais, debitos_locais
    )
    colunas_pareamento_saida = [
        "NR_TRAN_CREDITO", "FL_SANEADO", "NIVEL_EVIDENCIA",
        "QT_CANDIDATOS", "NR_TRAN_DEBITO_SELECIONADO",
        "NR_MCA_DEBITO_SELECIONADO", "DT_DEBITO_SELECIONADO",
        "VL_DEBITO_SELECIONADO", "DIFERENCA_DIAS",
        "MOTIVO_CLASSIFICACAO",
    ]
    df_pareamentos_entradas = pd.DataFrame(
        pareamentos_entradas,
        columns=colunas_pareamento_saida,
    )

    df_creditos_saida = df_creditos_efetivados[[
        "CD_CLI", "NR_TRAN_INST_PCT", "NR_PERIODO", "REF_PERIODO",
        "NR_MCA_PCT_OPB", "DT_TRAN", "CD_NTZ_CTB_TRAN",
        "CD_EST_TRAN_INST", "TX_EST_TRAN_INST", "CD_GR_CTGR_TRAN",
        "TX_DCR_GR_CTGR", "CD_CTGR_TRAN", "TX_DCR_CTGR_TRAN",
        "CD_TIP_MOE_CRR", "VL_TRAN", "TX_DCR_TRAN",
    ]].rename(columns={"NR_TRAN_INST_PCT": "NR_TRAN_CREDITO"})
    df_estudo_entradas = df_creditos_saida.merge(
        df_pareamentos_entradas,
        how="left",
        on="NR_TRAN_CREDITO",
        validate="one_to_one",
        sort=False,
    )
    COLUNAS_ESTUDO_ENTRADAS = [
        "CD_CLI", "NR_TRAN_CREDITO", "NR_PERIODO", "REF_PERIODO",
        "NR_MCA_PCT_OPB", "DT_TRAN", "CD_NTZ_CTB_TRAN",
        "CD_EST_TRAN_INST", "TX_EST_TRAN_INST", "CD_GR_CTGR_TRAN",
        "TX_DCR_GR_CTGR", "CD_CTGR_TRAN", "TX_DCR_CTGR_TRAN",
        "CD_TIP_MOE_CRR", "VL_TRAN", "TX_DCR_TRAN", "FL_SANEADO",
        "NIVEL_EVIDENCIA", "QT_CANDIDATOS",
        "NR_TRAN_DEBITO_SELECIONADO", "NR_MCA_DEBITO_SELECIONADO",
        "DT_DEBITO_SELECIONADO", "VL_DEBITO_SELECIONADO",
        "DIFERENCA_DIAS", "MOTIVO_CLASSIFICACAO",
    ]
    df_estudo_entradas = df_estudo_entradas[
        COLUNAS_ESTUDO_ENTRADAS
    ].copy()
    QT_CREDITOS_EFETIVADOS = int(len(creditos_locais))
    QT_LINHAS_ESTUDO_ENTRADAS = int(len(df_estudo_entradas))
except Exception as exc:
    df_estudo_entradas = None
    df_resumo_entradas = None
    if isinstance(exc, ErroContratoDados):
        raise
    raise ErroContratoDados(
        codigo="CONSTRUCAO_ESTUDO_ENTRADAS_FALHOU",
        mensagem="Falha ao construir o estudo das entradas.",
        etapa="ESTUDO_ENTRADAS",
        objeto="df_estudo_entradas",
        detalhes={"tipo_erro": type(exc).__name__},
    ) from exc


In [ ]:
try:
    qt_creditos_distintos = int(
        df_estudo_entradas["NR_TRAN_CREDITO"].nunique()
    )
    if (
        QT_LINHAS_ESTUDO_ENTRADAS != QT_CREDITOS_EFETIVADOS
        or qt_creditos_distintos != QT_CREDITOS_EFETIVADOS
    ):
        raise ErroContratoDados(
            codigo="GRAO_ESTUDO_ENTRADAS_INVALIDO",
            mensagem="df_estudo_entradas deve possuir uma linha por credito efetivado.",
            etapa="ESTUDO_ENTRADAS",
            objeto="NR_TRAN_CREDITO",
        )

    saneados = df_estudo_entradas.loc[
        df_estudo_entradas["FL_SANEADO"].eq("S")
    ]
    qt_debitos_reutilizados = int(
        saneados["NR_TRAN_DEBITO_SELECIONADO"].duplicated().sum()
    )
    if qt_debitos_reutilizados:
        raise ErroContratoDados(
            codigo="DEBITO_REUTILIZADO_NO_PAREAMENTO",
            mensagem="Um debito foi utilizado para sanear mais de um credito.",
            etapa="ESTUDO_ENTRADAS",
            objeto="NR_TRAN_DEBITO_SELECIONADO",
        )

    debitos_por_id = {
        linha["NR_TRAN_INST_PCT"]: linha
        for linha in df_debitos_efetivados.to_dict("records")
    }
    pares_invalidos = 0
    metadados_invalidos = 0
    motivos_por_dia = {
        0: ("MUITO_FORTE", "MATCH_MESMO_DIA"),
        1: ("FORTE", "MATCH_1_DIA"),
        2: ("FORTE", "MATCH_2_DIAS"),
        3: ("FORTE", "MATCH_3_DIAS"),
    }
    for linha in df_estudo_entradas.to_dict("records"):
        if linha["FL_SANEADO"] == "S":
            debito = debitos_por_id.get(linha["NR_TRAN_DEBITO_SELECIONADO"])
            if debito is None:
                pares_invalidos += 1
                continue
            diferenca = abs((linha["DT_TRAN"] - debito["DT_TRAN"]).days)
            esperado = motivos_por_dia.get(diferenca)
            par_valido = (
                linha["CD_CLI"] == debito["CD_CLI"]
                and linha["CD_NTZ_CTB_TRAN"] == "C"
                and linha["CD_EST_TRAN_INST"] == 0
                and debito["CD_NTZ_CTB_TRAN"] == "D"
                and debito["CD_EST_TRAN_INST"] == 0
                and linha["VL_TRAN"] == debito["VL_TRAN"]
                and linha["CD_TIP_MOE_CRR"] == debito["CD_TIP_MOE_CRR"]
                and linha["NR_MCA_PCT_OPB"] != debito["NR_MCA_PCT_OPB"]
                and linha["DIFERENCA_DIAS"] == diferenca
                and linha["NR_MCA_DEBITO_SELECIONADO"] == debito["NR_MCA_PCT_OPB"]
                and linha["DT_DEBITO_SELECIONADO"] == debito["DT_TRAN"]
                and linha["VL_DEBITO_SELECIONADO"] == debito["VL_TRAN"]
                and esperado is not None
                and linha["NIVEL_EVIDENCIA"] == esperado[0]
                and linha["MOTIVO_CLASSIFICACAO"] == esperado[1]
                and int(linha["QT_CANDIDATOS"]) >= 1
            )
            pares_invalidos += int(not par_valido)
        elif linha["FL_SANEADO"] == "N":
            campos_nulos = (
                "NR_TRAN_DEBITO_SELECIONADO",
                "NR_MCA_DEBITO_SELECIONADO",
                "DT_DEBITO_SELECIONADO",
                "VL_DEBITO_SELECIONADO",
                "DIFERENCA_DIAS",
                "NIVEL_EVIDENCIA",
            )
            valido = (
                all(pd.isna(linha[campo]) for campo in campos_nulos)
                and int(linha["QT_CANDIDATOS"]) == 0
                and linha["MOTIVO_CLASSIFICACAO"] == "SEM_DEBITO_DISPONIVEL"
            )
            metadados_invalidos += int(not valido)
        else:
            metadados_invalidos += 1

    if pares_invalidos:
        raise ErroContratoDados(
            codigo="PAR_PAREAMENTO_ENTRADAS_INVALIDO",
            mensagem="Um par selecionado nao respeita as regras do estudo.",
            etapa="ESTUDO_ENTRADAS",
            objeto="pareamento credito-debito",
            detalhes={"qt_afetada": pares_invalidos},
        )
    if metadados_invalidos:
        raise ErroContratoDados(
            codigo="METADADO_ESTUDO_ENTRADAS_INVALIDO",
            mensagem="A classificacao do estudo possui metadados inconsistentes.",
            etapa="ESTUDO_ENTRADAS",
            objeto="df_estudo_entradas",
            detalhes={"qt_afetada": metadados_invalidos},
        )

    chaves_resumo_entradas = [
        "CD_CLI", "NR_PERIODO", "REF_PERIODO", "CD_TIP_MOE_CRR"
    ]
    linhas_resumo_entradas = []
    for chave, grupo in df_estudo_entradas.groupby(
        chaves_resumo_entradas, dropna=False, sort=False
    ):
        grupo_saneado = grupo.loc[grupo["FL_SANEADO"].eq("S")]
        identificadas = sum(grupo["VL_TRAN"], Decimal("0"))
        valor_proprio = sum(grupo_saneado["VL_TRAN"], Decimal("0"))
        corrigidas = identificadas - valor_proprio
        if valor_proprio < 0 or valor_proprio > identificadas or corrigidas < 0:
            raise ErroContratoDados(
                codigo="RESUMO_ENTRADAS_INCONSISTENTE",
                mensagem="O resumo de entradas violou limites financeiros.",
                etapa="ESTUDO_ENTRADAS",
            )
        if identificadas != valor_proprio + corrigidas:
            raise ErroContratoDados(
                codigo="RESUMO_ENTRADAS_INCONSISTENTE",
                mensagem="O resumo de entradas nao reconciliou.",
                etapa="ESTUDO_ENTRADAS",
            )
        linhas_resumo_entradas.append({
            "CD_CLI": chave[0],
            "NR_PERIODO": chave[1],
            "REF_PERIODO": chave[2],
            "CD_TIP_MOE_CRR": chave[3],
            "QT_ENTRADAS": int(len(grupo)),
            "ENTRADAS_TOTAIS_IDENTIFICADAS": identificadas,
            "QT_MOVIMENTACOES_PROPRIAS": int(len(grupo_saneado)),
            "VALOR_TRANSACAO_PROPRIA": valor_proprio,
            "ENTRADAS_TOTAIS_CORRIGIDAS": corrigidas,
        })

    COLUNAS_RESUMO_ENTRADAS = [
        *chaves_resumo_entradas,
        "QT_ENTRADAS", "ENTRADAS_TOTAIS_IDENTIFICADAS",
        "QT_MOVIMENTACOES_PROPRIAS", "VALOR_TRANSACAO_PROPRIA",
        "ENTRADAS_TOTAIS_CORRIGIDAS",
    ]
    df_resumo_entradas = pd.DataFrame(
        linhas_resumo_entradas,
        columns=COLUNAS_RESUMO_ENTRADAS,
    )

    # Reconciliacao detalhe -> periodo/moeda -> total da mesma moeda.
    total_detalhe_moeda = {}
    for moeda, grupo in df_estudo_entradas.groupby(
        "CD_TIP_MOE_CRR", dropna=False, sort=False
    ):
        total_detalhe_moeda[moeda] = {
            "qt": int(len(grupo)),
            "identificadas": sum(grupo["VL_TRAN"], Decimal("0")),
            "proprias": sum(
                grupo.loc[grupo["FL_SANEADO"].eq("S"), "VL_TRAN"],
                Decimal("0"),
            ),
        }
    total_resumo_moeda = {}
    for moeda, grupo in df_resumo_entradas.groupby(
        "CD_TIP_MOE_CRR", dropna=False, sort=False
    ):
        total_resumo_moeda[moeda] = {
            "qt": int(grupo["QT_ENTRADAS"].sum()),
            "identificadas": sum(
                grupo["ENTRADAS_TOTAIS_IDENTIFICADAS"], Decimal("0")
            ),
            "proprias": sum(
                grupo["VALOR_TRANSACAO_PROPRIA"], Decimal("0")
            ),
        }
    if total_detalhe_moeda != total_resumo_moeda:
        raise ErroContratoDados(
            codigo="RECONCILIACAO_TOTAL_MOEDA_FALHOU",
            mensagem="Detalhe e resumo divergem na mesma moeda.",
            etapa="ESTUDO_ENTRADAS",
        )

    print("### ESTUDO_ENTRADAS")
    display(df_estudo_entradas.sort_values(
        ["NR_PERIODO", "DT_TRAN", "NR_TRAN_CREDITO"], kind="mergesort"
    ))
    print("### RESUMO_ENTRADAS")
    display(df_resumo_entradas.sort_values(
        ["NR_PERIODO", "CD_TIP_MOE_CRR"], kind="mergesort"
    ))
except Exception as exc:
    df_estudo_entradas = None
    df_resumo_entradas = None
    if isinstance(exc, ErroContratoDados):
        raise
    raise ErroContratoDados(
        codigo="VALIDACAO_ESTUDO_ENTRADAS_FALHOU",
        mensagem="Falha ao validar ou resumir o estudo das entradas.",
        etapa="ESTUDO_ENTRADAS",
        objeto="df_resumo_entradas",
        detalhes={"tipo_erro": type(exc).__name__},
    ) from exc


## 8. Classificacao Radar

O mapa Radar permanece internalizado com 71 categorias e duas naturezas por categoria. A categoria vigente e a natureza contabil formam a chave da classificacao.


In [ ]:
df_radar = None

try:
    codigos_radar_credito = {
            "Entrada": 0,
            "Renda": 1,
            "Estorno": 2,
            "Resgate": 3,
            "Crédito": 4,
        }

    codigos_radar_debito = {
            "Outros": 5,
            "Essenciais": 6,
            "Flexíveis": 7,
            "Futuro": 8,
            "Dívidas": 9,
        }

    base_mapa_radar = [
            (0, "Entrada", "Outros", "N"),
            (83, "Entrada", "Outros", "N"),
            (1, "Renda", "Outros", "N"),
            (2, "Renda", "Outros", "N"),
            (3, "Estorno", "Outros", "N"),
            (4, "Renda", "Outros", "N"),
            (5, "Renda", "Outros", "N"),
            (6, "Estorno", "Essenciais", "N"),
            (7, "Estorno", "Essenciais", "N"),
            (9, "Crédito", "Dívidas", "N"),
            (10, "Estorno", "Essenciais", "N"),
            (11, "Estorno", "Flexíveis", "N"),
            (12, "Estorno", "Essenciais", "N"),
            (13, "Estorno", "Flexíveis", "N"),
            (14, "Estorno", "Flexíveis", "N"),
            (3790, "Estorno", "Flexíveis", "N"),
            (15, "Estorno", "Flexíveis", "N"),
            (16, "Estorno", "Essenciais", "N"),
            (17, "Estorno", "Flexíveis", "N"),
            (18, "Estorno", "Flexíveis", "N"),
            (20, "Estorno", "Flexíveis", "N"),
            (21, "Estorno", "Flexíveis", "N"),
            (22, "Estorno", "Flexíveis", "N"),
            (25, "Estorno", "Flexíveis", "N"),
            (26, "Estorno", "Flexíveis", "N"),
            (61, "Estorno", "Flexíveis", "N"),
            (27, "Estorno", "Essenciais", "N"),
            (28, "Estorno", "Essenciais", "N"),
            (29, "Estorno", "Essenciais", "N"),
            (30, "Estorno", "Essenciais", "N"),
            (32, "Estorno", "Essenciais", "N"),
            (35, "Estorno", "Flexíveis", "N"),
            (36, "Crédito", "Dívidas", "N"),
            (37, "Estorno", "Essenciais", "N"),
            (38, "Estorno", "Flexíveis", "N"),
            (39, "Estorno", "Flexíveis", "N"),
            (40, "Estorno", "Flexíveis", "N"),
            (41, "Estorno", "Essenciais", "N"),
            (42, "Estorno", "Flexíveis", "N"),
            (43, "Estorno", "Flexíveis", "N"),
            (44, "Estorno", "Outros", "N"),
            (45, "Estorno", "Essenciais", "N"),
            (46, "Estorno", "Outros", "N"),
            (47, "Estorno", "Flexíveis", "N"),
            (48, "Estorno", "Flexíveis", "N"),
            (49, "Estorno", "Flexíveis", "N"),
            (60, "Estorno", "Flexíveis", "N"),
            (4417, "Crédito", "Dívidas", "N"),
            (51, "Estorno", "Essenciais", "N"),
            (53, "Estorno", "Flexíveis", "N"),
            (54, "Estorno", "Essenciais", "N"),
            (55, "Estorno", "Essenciais", "N"),
            (56, "Estorno", "Essenciais", "N"),
            (57, "Estorno", "Essenciais", "N"),
            (58, "Estorno", "Futuro", "N"),
            (59, "Estorno", "Dívidas", "N"),
            (3787, "Estorno", "Dívidas", "N"),
            (3788, "Estorno", "Dívidas", "N"),
            (279, "Entrada", "Outros", "N"),
            (39434, "Entrada", "Outros", "N"),
            (39435, "Entrada", "Outros", "N"),
            (39436, "Entrada", "Outros", "N"),
            (39437, "Estorno", "Outros", "N"),
            (111, "Estorno", "Dívidas", "N"),
            (448977, "Estorno", "Futuro", "N"),
            (448978, "Resgate", "Outros", "N"),
            (300, "Renda", "Outros", "S"),
            (310, "Renda", "Outros", "S"),
            (330, "Renda", "Outros", "S"),
            (350, "Estorno", "Outros", "S"),
            (370, "Estorno", "Outros", "S"),
        ]

    linhas_mapa_radar = []
    for cd_categoria, tx_credito, tx_debito, fl_agro in base_mapa_radar:
        linhas_mapa_radar.append({
            "CD_CTGR_TRAN": cd_categoria,
            "CD_NTZ_CTB_TRAN": "C",
            "CD_CLASSIFICACAO_RADAR": codigos_radar_credito[tx_credito],
            "TX_DCR_CLASSIFICACAO_RADAR": tx_credito,
            "FL_AGRO": fl_agro,
        })
        linhas_mapa_radar.append({
            "CD_CTGR_TRAN": cd_categoria,
            "CD_NTZ_CTB_TRAN": "D",
            "CD_CLASSIFICACAO_RADAR": codigos_radar_debito[tx_debito],
            "TX_DCR_CLASSIFICACAO_RADAR": tx_debito,
            "FL_AGRO": fl_agro,
        })

    df_mapa_radar = pd.DataFrame(linhas_mapa_radar)
    qt_chaves_mapa = int(
        df_mapa_radar[["CD_CTGR_TRAN", "CD_NTZ_CTB_TRAN"]]
        .drop_duplicates()
        .shape[0]
    )
    if (
        len(base_mapa_radar) != 71
        or len(df_mapa_radar) != 142
        or qt_chaves_mapa != 142
    ):
        raise ErroContratoDados(
            codigo="RADAR_MAPA_INVALIDO",
            mensagem="O mapa Radar embutido nao possui as 142 chaves esperadas.",
            etapa="RADAR",
            objeto="df_mapa_radar",
        )

    mascara_mapa_invalido = ~(
        (
            df_mapa_radar["CD_NTZ_CTB_TRAN"].eq("C")
            & df_mapa_radar["CD_CLASSIFICACAO_RADAR"].between(0, 4)
        )
        | (
            df_mapa_radar["CD_NTZ_CTB_TRAN"].eq("D")
            & df_mapa_radar["CD_CLASSIFICACAO_RADAR"].between(5, 9)
        )
    ) | ~df_mapa_radar["FL_AGRO"].isin(["S", "N"])
    if mascara_mapa_invalido.any():
        raise ErroContratoDados(
            codigo="RADAR_MAPA_CLASSIFICACAO_INVALIDA",
            mensagem="O mapa Radar possui classificacao ou FL_AGRO invalido.",
            etapa="RADAR",
            objeto="df_mapa_radar",
        )

    exemplo_1 = df_mapa_radar.loc[
        df_mapa_radar["CD_CTGR_TRAN"].eq(1)
        & df_mapa_radar["CD_NTZ_CTB_TRAN"].eq("C")
    ].iloc[0]
    exemplo_32 = df_mapa_radar.loc[
        df_mapa_radar["CD_CTGR_TRAN"].eq(32)
        & df_mapa_radar["CD_NTZ_CTB_TRAN"].eq("D")
    ].iloc[0]
    categorias_agro = {300, 310, 330, 350, 370}
    flags_agro_validas = all(
        (linha["FL_AGRO"] == "S")
        == (linha["CD_CTGR_TRAN"] in categorias_agro)
        for linha in df_mapa_radar.to_dict("records")
    )
    if not (
        exemplo_1["CD_CLASSIFICACAO_RADAR"] == 1
        and exemplo_1["TX_DCR_CLASSIFICACAO_RADAR"] == "Renda"
        and exemplo_1["FL_AGRO"] == "N"
        and exemplo_32["CD_CLASSIFICACAO_RADAR"] == 6
        and exemplo_32["TX_DCR_CLASSIFICACAO_RADAR"] == "Essenciais"
        and exemplo_32["FL_AGRO"] == "N"
        and flags_agro_validas
    ):
        raise ErroContratoDados(
            codigo="RADAR_MAPA_CONTEUDO_DIVERGENTE",
            mensagem="O mapa Radar diverge dos casos de controle.",
            etapa="RADAR",
            objeto="df_mapa_radar",
        )
except Exception as exc:
    df_radar = None
    if isinstance(exc, ErroContratoDados):
        raise
    raise ErroContratoDados(
        codigo="RADAR_MAPA_CARREGAMENTO_FALHOU",
        mensagem="Falha ao preparar o mapa embutido do Radar.",
        etapa="RADAR",
        objeto="df_mapa_radar",
        detalhes={"tipo_erro": type(exc).__name__},
    ) from exc


In [ ]:
try:
    df_saneados_radar = df_estudo_entradas.loc[
        df_estudo_entradas["FL_SANEADO"].eq("S")
    ]
    ids_creditos_proprios = df_saneados_radar[
        "NR_TRAN_CREDITO"
    ].tolist()
    ids_debitos_proprios = df_saneados_radar[
        "NR_TRAN_DEBITO_SELECIONADO"
    ].tolist()
    if len(ids_creditos_proprios) != len(set(ids_creditos_proprios)):
        raise ErroContratoDados(
            codigo="RADAR_CREDITO_PROPRIO_DUPLICADO",
            mensagem="Um credito saneado aparece mais de uma vez.",
            etapa="RADAR",
            objeto="NR_TRAN_CREDITO",
        )
    if len(ids_debitos_proprios) != len(set(ids_debitos_proprios)):
        raise ErroContratoDados(
            codigo="RADAR_DEBITO_PROPRIO_DUPLICADO",
            mensagem="Um debito selecionado aparece mais de uma vez.",
            etapa="RADAR",
            objeto="NR_TRAN_DEBITO_SELECIONADO",
        )
    ids_movimentacao_propria = set(ids_creditos_proprios) | set(
        ids_debitos_proprios
    )
    if len(ids_movimentacao_propria) != (
        len(ids_creditos_proprios) + len(ids_debitos_proprios)
    ):
        raise ErroContratoDados(
            codigo="RADAR_MOVIMENTACAO_PROPRIA_DUPLICADA",
            mensagem="Um identificador foi marcado simultaneamente mais de uma vez.",
            etapa="RADAR",
        )

    transacoes_radar = df_enriquecido.copy()
    transacoes_radar["_NATUREZA_RADAR"] = (
        transacoes_radar["CD_NTZ_CTB_TRAN"]
        .astype("string").str.strip().str.upper()
    )
    transacoes_radar["_CHAVE_CTGR_RADAR"] = (
        transacoes_radar["CD_CTGR_TRAN"]
        .map(inteiro_exato)
        .astype("Int64")
    )
    mapa_radar_join = df_mapa_radar.rename(
        columns={"CD_NTZ_CTB_TRAN": "_NATUREZA_MAPA_RADAR"}
    )
    mapa_radar_join["_CHAVE_CTGR_RADAR"] = (
        mapa_radar_join["CD_CTGR_TRAN"]
        .map(inteiro_exato)
        .astype("Int64")
    )
    mapa_radar_join = mapa_radar_join.drop(columns=["CD_CTGR_TRAN"])
    try:
        radar_enriquecido = transacoes_radar.merge(
            mapa_radar_join,
            how="left",
            left_on=["_CHAVE_CTGR_RADAR", "_NATUREZA_RADAR"],
            right_on=["_CHAVE_CTGR_RADAR", "_NATUREZA_MAPA_RADAR"],
            validate="many_to_one",
            indicator="_COBERTURA_RADAR",
            sort=False,
        )
    except pd.errors.MergeError as exc:
        raise ErroContratoDados(
            codigo="RADAR_MULTIPLICACAO_NO_JOIN",
            mensagem="O enriquecimento Radar multiplicaria transacoes.",
            etapa="RADAR",
            objeto="df_mapa_radar",
        ) from exc

    radar_enriquecido["FL_MOVIMENTACAO_PROPRIA"] = (
        radar_enriquecido["NR_TRAN_INST_PCT"]
        .isin(ids_movimentacao_propria)
        .map({True: "S", False: "N"})
    )
    COLUNAS_RADAR = [
        "NR_TRAN_INST_PCT", "CD_CLI", "NR_PERIODO", "REF_PERIODO",
        "NR_MCA_PCT_OPB", "DT_TRAN", "CD_NTZ_CTB_TRAN",
        "CD_EST_TRAN_INST", "TX_EST_TRAN_INST", "CD_GR_CTGR_TRAN",
        "TX_DCR_GR_CTGR", "CD_CTGR_TRAN", "TX_DCR_CTGR_TRAN",
        "CD_TIP_MOE_CRR", "VL_TRAN", "TX_DCR_TRAN",
        "CD_CLASSIFICACAO_RADAR", "TX_DCR_CLASSIFICACAO_RADAR",
        "FL_AGRO", "FL_MOVIMENTACAO_PROPRIA",
    ]
    df_radar = radar_enriquecido[COLUNAS_RADAR].copy()

    QT_LINHAS_DF_RADAR = int(len(df_radar))
    QT_IDS_DISTINTOS_DF_RADAR = int(
        df_radar["NR_TRAN_INST_PCT"].nunique()
    )
    if QT_LINHAS_DF_RADAR > QT_TRANSACOES_ENRIQUECIDAS:
        raise ErroContratoDados(
            codigo="RADAR_MULTIPLICACAO_NO_JOIN",
            mensagem="O enriquecimento Radar multiplicou transacoes.",
            etapa="RADAR",
        )
    if QT_LINHAS_DF_RADAR < QT_TRANSACOES_ENRIQUECIDAS:
        raise ErroContratoDados(
            codigo="RADAR_PERDA_NO_JOIN",
            mensagem="O enriquecimento Radar perdeu transacoes.",
            etapa="RADAR",
        )
    if (
        QT_IDS_DISTINTOS_DF_RADAR != QT_IDS_ENRIQUECIDOS
        or QT_LINHAS_DF_RADAR != QT_IDS_DISTINTOS_DF_RADAR
    ):
        raise ErroContratoDados(
            codigo="RADAR_GRAO_INVALIDO",
            mensagem="df_radar deve possuir uma linha por NR_TRAN_INST_PCT.",
            etapa="RADAR",
            objeto="NR_TRAN_INST_PCT",
        )

    mascara_nao_mapeada = radar_enriquecido["_COBERTURA_RADAR"].ne("both")
    if mascara_nao_mapeada.any():
        raise ErroContratoDados(
            codigo="RADAR_CATEGORIA_NAO_MAPEADA",
            mensagem="Uma categoria vigente e natureza nao existe no mapa Radar.",
            etapa="RADAR",
            objeto="CD_CTGR_TRAN + CD_NTZ_CTB_TRAN",
            detalhes={"qt_afetada": int(mascara_nao_mapeada.sum())},
        )

    mascara_classificacao_invalida = ~(
        (
            df_radar["CD_NTZ_CTB_TRAN"].eq("C")
            & df_radar["CD_CLASSIFICACAO_RADAR"].between(0, 4)
        )
        | (
            df_radar["CD_NTZ_CTB_TRAN"].eq("D")
            & df_radar["CD_CLASSIFICACAO_RADAR"].between(5, 9)
        )
    )
    if mascara_classificacao_invalida.any():
        raise ErroContratoDados(
            codigo="RADAR_CLASSIFICACAO_INVALIDA",
            mensagem="Uma classificacao Radar nao respeita a faixa da natureza.",
            etapa="RADAR",
        )
    mascara_flag_invalida = (
        ~df_radar["FL_AGRO"].isin(["S", "N"])
        | ~df_radar["FL_MOVIMENTACAO_PROPRIA"].isin(["S", "N"])
    )
    if mascara_flag_invalida.any():
        raise ErroContratoDados(
            codigo="RADAR_FLAG_INVALIDA",
            mensagem="As flags do Radar devem possuir somente S ou N.",
            etapa="RADAR",
        )

    qt_creditos_marcados = int((
        df_radar["CD_NTZ_CTB_TRAN"].eq("C")
        & df_radar["FL_MOVIMENTACAO_PROPRIA"].eq("S")
    ).sum())
    qt_debitos_marcados = int((
        df_radar["CD_NTZ_CTB_TRAN"].eq("D")
        & df_radar["FL_MOVIMENTACAO_PROPRIA"].eq("S")
    ).sum())
    if qt_creditos_marcados != len(ids_creditos_proprios):
        raise ErroContratoDados(
            codigo="RADAR_QUANTIDADE_CREDITOS_PROPRIOS_DIVERGENTE",
            mensagem="Creditos proprios divergem do estudo de entradas.",
            etapa="RADAR",
        )
    if qt_debitos_marcados != len(ids_debitos_proprios):
        raise ErroContratoDados(
            codigo="RADAR_QUANTIDADE_DEBITOS_PROPRIOS_DIVERGENTE",
            mensagem="Debitos proprios divergem do estudo de entradas.",
            etapa="RADAR",
        )

    chaves_reconciliacao = [
        "CD_CLI", "NR_PERIODO", "REF_PERIODO", "CD_TIP_MOE_CRR"
    ]
    valores_radar = {}
    creditos_proprios_radar = df_radar.loc[
        df_radar["CD_NTZ_CTB_TRAN"].eq("C")
        & df_radar["FL_MOVIMENTACAO_PROPRIA"].eq("S")
    ]
    for chave, grupo in creditos_proprios_radar.groupby(
        chaves_reconciliacao, dropna=False, sort=False
    ):
        valores_radar[chave] = sum(grupo["VL_TRAN"], Decimal("0"))
    valores_estudo = {
        tuple(linha[chave] for chave in chaves_reconciliacao): linha[
            "VALOR_TRANSACAO_PROPRIA"
        ]
        for linha in df_resumo_entradas.to_dict("records")
    }
    todas_chaves = set(valores_radar) | set(valores_estudo)
    divergencias = sum(
        valores_radar.get(chave, Decimal("0"))
        != valores_estudo.get(chave, Decimal("0"))
        for chave in todas_chaves
    )
    if divergencias:
        raise ErroContratoDados(
            codigo="RADAR_VALOR_CREDITOS_PROPRIOS_DIVERGENTE",
            mensagem="Valores proprios do Radar divergem do resumo.",
            etapa="RADAR",
            detalhes={"qt_afetada": divergencias},
        )

    print("### RADAR")
    display(df_radar.sort_values(
        ["NR_PERIODO", "DT_TRAN", "NR_TRAN_INST_PCT"], kind="mergesort"
    ))
except Exception as exc:
    df_radar = None
    if isinstance(exc, ErroContratoDados):
        raise
    raise ErroContratoDados(
        codigo="RADAR_PROCESSAMENTO_FALHOU",
        mensagem="Falha ao construir ou validar a camada Radar.",
        etapa="RADAR",
        objeto="df_radar",
        detalhes={"tipo_erro": type(exc).__name__},
    ) from exc


In [ ]:
# Dashboard Financeiro V1 - consumo direto dos DataFrames locais.
from src.app import fluxo_dashboard

resultado_dashboard = fluxo_dashboard(
    df_radar,
    df_estudo_entradas,
    df_resumo_entradas,
    df_periodos_utilizados,
    df_resumo_execucao,
    exibir=True,
)
